# ClinicalRAG — RAGAS Evaluation Notebook
**Project**: clinical-note-rag-intelligence · MIMIC-IV Clinical Notes RAG  
**Purpose**: Evaluate RAG pipeline quality using the RAGAS framework with auto-generated test sets

---

## Evaluation Strategy

Questions and ground-truth answers are automatically generated directly from MIMIC-IV discharge notes using **RAGAS `TestsetGenerator`**, so evaluation coverage is not limited to a fixed hand-crafted dataset.

**RAGAS Four Metrics**:

| Metric | What it measures | Requires ground_truth? |
|--------|-----------------|------------------------|
| **Faithfulness** | Are answer claims supported by retrieved context? (hallucination detection) | No |
| **Answer Relevancy** | Is the answer on-topic and complete? | No |
| **Context Precision** | Are retrieved documents relevant to the question? | Yes |
| **Context Recall** | Does retrieval capture all required information? | Yes |

## Cell 1 — Install Dependencies

Install required packages. `ragas==0.2.9` is the tested stable version — do not upgrade arbitrarily as the API may change.

In [ ]:
# Run once to install required packages. Skip if already installed.
import subprocess, sys
packages = [
    "ragas==0.2.9", "datasets", "openai", "openpyxl", "pandas",
    "langchain-openai", "langchain-community", "matplotlib", "seaborn", "faiss-cpu",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages installed.")


## Cell 2 — Global Imports

Import all required libraries and suppress unnecessary warnings to keep output clean.

In [ ]:
import os, sys, json, time, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

# Make src/ importable from the project root
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv()

# RAGAS
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document

print("Imports complete.")


## Cell 3 — Configuration (**only cell you need to modify**)

- `OPENAI_API_KEY`: your OpenAI API key (or set `OPENAI_API_KEY=sk-...` in a `.env` file)
- `DISCHARGE_CSV_PATH`: path to the raw MIMIC-IV discharge notes CSV for the evaluation
- `NROWS`: how many notes to read from CSV (more = more stable generation, higher token cost)
- `TESTSET_SIZE`: number of Q&A pairs to auto-generate (5~15 recommended)
- `SLEEP_BETWEEN_CALLS`: seconds between API calls to avoid rate limiting


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  USER CONFIG — the only cell you need to modify
# ─────────────────────────────────────────────────────────────────
import os
OPENAI_API_KEY      = os.getenv("OPENAI_API_KEY")
CHAT_MODEL          = os.getenv("OPENAI_CHAT_MODEL",  "gpt-4o-mini")
EMBED_MODEL         = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

# Path to MIMIC-IV discharge notes CSV (or the included synthetic sample)
DISCHARGE_CSV_PATH  = os.getenv("DISCHARGE_CSV_PATH", "./data/sample/discharge_sample.csv")
TESTSET_SIZE        = int(os.getenv("TESTSET_SIZE", 10))   # Q&A pairs to auto-generate
NROWS               = int(os.getenv("NROWS", 50))          # notes to read from CSV
OUTPUT_DIR          = os.getenv("OUTPUT_DIR", "./evaluation_results")
SLEEP_BETWEEN_CALLS = float(os.getenv("SLEEP_BETWEEN_CALLS", 1.5))

# ── Validation ────────────────────────────────────────────────────────────
assert OPENAI_API_KEY, "Set OPENAI_API_KEY in a .env file or environment variable."
assert os.path.exists(DISCHARGE_CSV_PATH), (
    f"Discharge CSV not found: {DISCHARGE_CSV_PATH}\n"
    "Set DISCHARGE_CSV_PATH in your .env to point to discharge.csv."
)

from src.config import config
assert os.path.exists(config.FAISS_INDEX_DIR), (
    f"FAISS index not found at: {config.FAISS_INDEX_DIR}\n"
    "Run: python -c 'from src.ingest import build_index; build_index()'"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Chat model     : {CHAT_MODEL}")
print(f"  Embed model    : {EMBED_MODEL}")
print(f"  Discharge CSV  : {DISCHARGE_CSV_PATH}")
print(f"  Output dir     : {OUTPUT_DIR}")
print(f"  FAISS index    : {config.FAISS_INDEX_DIR}")


## Cell 4 — Build RAGAS LLM & Embeddings

Each RAGAS metric internally calls an LLM (as judge) and Embeddings (for semantic similarity).  
RAGAS requires LangChain models to be wrapped before passing in. Built once, shared by both modes.


In [ ]:
def build_ragas_llm_and_embeddings():
    llm = LangchainLLMWrapper(
        ChatOpenAI(
            model=CHAT_MODEL,
            temperature=0
        )
    )

    embeddings = LangchainEmbeddingsWrapper(
        OpenAIEmbeddings(
            model=EMBED_MODEL
        )
    )

    return llm, embeddings


def inject_llm_into_metrics(llm, embeddings):
    """Assign LLM and Embeddings to each RAGAS metric before evaluate()."""
    for metric in [faithfulness, answer_relevancy]:
        metric.llm = llm
        if hasattr(metric, "embeddings"):
            metric.embeddings = embeddings


ragas_llm, ragas_embeddings = build_ragas_llm_and_embeddings()
inject_llm_into_metrics(ragas_llm, ragas_embeddings)
print("RAGAS LLM and Embeddings initialized.")


## Cell 5 — Shared Helper: Run RAG Pipeline on a Batch of Questions

This function sends a batch of questions through the RAG pipeline, collecting each question's **answer** and **retrieved documents** (`contexts`), formatted for RAGAS:

```
{question, answer, contexts, ground_truth}
```


In [ ]:
from src.pipeline import run_pipeline


def run_pipeline_batch(questions: list, ground_truths: list) -> list:
    """
    Execute the full RAG pipeline for each question and collect
    RAGAS-formatted records.

    Parameters
    ----------
    questions     : list of question strings
    ground_truths : list of reference answer strings (same length)

    Returns
    -------
    List of dicts: {question, answer, contexts, ground_truth}
    """
    assert len(questions) == len(ground_truths), \
        "questions and ground_truths must have equal length."

    records = []
    total   = len(questions)

    for i, (q, gt) in enumerate(zip(questions, ground_truths), start=1):
        preview = q[:85] + "..." if len(q) > 85 else q
        print(f"  [{i:02d}/{total}] {preview}")

        try:
            result = run_pipeline(q)

            answer = result.answer or "Unable to confirm from the available retrieved notes."

            # Prefer reranked docs; fall back to retrieved docs
            docs_source = result.reranked_docs or result.retrieved_docs
            contexts = [
                d.get("content", "").strip()
                for d in docs_source
                if d.get("content", "").strip()
            ]
            if not contexts:
                contexts = ["[No context retrieved by the pipeline]"]

            print(f"        answer={len(answer)} chars | contexts={len(contexts)} chunks")

        except Exception as e:
            print(f"        Pipeline error: {e}")
            answer   = "[Pipeline error]"
            contexts = ["[Pipeline error — no context retrieved]"]

        records.append({
            "question":     q,
            "answer":       answer,
            "contexts":     contexts,
            "ground_truth": gt,
        })

        time.sleep(SLEEP_BETWEEN_CALLS)

    return records


print("Pipeline batch runner defined.")


## Cell 6 — Shared Helper: Run RAGAS Evaluation

Accepts pipeline output records, calls RAGAS to compute all four metrics, and prints a visual progress bar summary.


In [ ]:
def run_ragas(records: list, label: str = ""):
    """
    Run RAGAS evaluation on pipeline output records.

    Parameters
    ----------
    records : output of run_pipeline_batch()
    label   : display label for console output

    Returns
    -------
    (ragas_result, detail_df, scores_dict)
    """
    dataset = Dataset.from_list([
        {
            "question":     r["question"],
            "answer":       r["answer"],
            "contexts":     r["contexts"],
            "ground_truth": r["ground_truth"],
        }
        for r in records
    ])

    print(f"\nRunning RAGAS [{label}] on {len(records)} samples...")

    ragas_result = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy],
    )

    detail_df = ragas_result.to_pandas()
    scores = {
        "faithfulness": round(np.mean(ragas_result["faithfulness"]), 4),
        "answer_relevancy": round(np.mean(ragas_result["answer_relevancy"]), 4),
    }

    print(f"\n{'─'*55}")
    print(f"  RAGAS Results — {label}")
    print(f"{'─'*55}")
    for k, v in scores.items():
        bar = chr(9608) * int(v * 25)
        print(f"  {k:<22} {v:.4f}  |{bar:<25}|")
    print(f"{'─'*55}")

    return ragas_result, detail_df, scores


print("RAGAS evaluation helper defined.")


---
# ══════════════════════════════════════════════════════
# RAGAS EVALUATION — Auto-Generated Test Set
# ══════════════════════════════════════════════════════

**Pipeline**:
1. Load raw MIMIC-IV discharge notes from `discharge.csv`
2. Convert to LangChain `Document` objects and pass to **RAGAS `TestsetGenerator`**
3. `TestsetGenerator` calls GPT to **automatically read documents and generate N {question, ground_truth} pairs**
4. Send auto-generated questions through the full RAG pipeline to collect answers and contexts
5. Evaluate with RAGAS across all four metrics

**Advantage**: No hard-coded QA dataset needed — questions are grounded in the actual documents, giving unbiased coverage of arbitrary clinical inputs.

## Cell 7 — Load Raw Discharge Notes


In [ ]:
# ── Load raw MIMIC-IV discharge notes from CSV ────────────────────────────────
# Supported columns: 'text' (required), 'note_id' (optional)
# discharge.csv typically contains columns: subject_id, hadm_id, note_id, charttime, text, ...

df_notes = pd.read_csv(
    DISCHARGE_CSV_PATH,
    nrows=NROWS,
    usecols=lambda c: c in {"note_id", "subject_id", "hadm_id", "text"},
)

# Identify the text column (handle alternate naming)
text_col = "text" if "text" in df_notes.columns else df_notes.columns[-1]
df_notes = df_notes.dropna(subset=[text_col]).reset_index(drop=True)

raw_documents = [
    Document(
        page_content=str(row[text_col]).strip(),
        metadata={
            "note_id":    str(row.get("note_id", f"note_{i}")),
            "subject_id": str(row.get("subject_id", "")),
            "hadm_id":    str(row.get("hadm_id", "")),
            "source":     "discharge.csv",
        },
    )
    for i, row in df_notes.iterrows()
    if len(str(row[text_col]).strip()) > 50  # skip near-empty notes
]

print(f"Loaded {len(raw_documents)} discharge notes from CSV.")
avg_len = sum(len(d.page_content) for d in raw_documents) // max(len(raw_documents), 1)
print(f"  Avg note length : {avg_len} chars")
print(f"  Sample note IDs : {[d.metadata['note_id'] for d in raw_documents[:5]]}")


## Cell 12 — Auto-Generate Test Set with RAGAS TestsetGenerator

`TestsetGenerator` workflow:
1. Reads all documents, uses LLM to extract "knowledge nodes"
2. Designs questions targeting each node
3. Uses LLM to answer from source text, producing ground_truth
4. Returns N `{user_input, reference}` pairs

With `TESTSET_SIZE=8` expect 2–5 minutes and additional OpenAI token usage.


In [ ]:
ragas_llm, ragas_embeddings = build_ragas_llm_and_embeddings()
inject_llm_into_metrics(ragas_llm, ragas_embeddings)

In [ ]:
print(f"Initializing TestsetGenerator (will generate {TESTSET_SIZE} test cases)...")
print("This calls OpenAI multiple times and may take 2-5 minutes.\n")

generator = TestsetGenerator(
    llm=ragas_llm,
    embedding_model=ragas_embeddings,
)

generated_testset = generator.generate_with_langchain_docs(
    documents=raw_documents,
    testset_size=TESTSET_SIZE,
)

df_generated = generated_testset.to_pandas()

# Normalize column names across RAGAS versions
q_col  = "user_input" if "user_input" in df_generated.columns else "question"
gt_col = "reference"  if "reference"  in df_generated.columns else "ground_truth"

print(f"Generated {len(df_generated)} test cases.")
print(f"Columns: {list(df_generated.columns)}\n")
df_generated[[q_col, gt_col]].head(5)


## Cell 13 — Inspect & Save Generated Test Set

**Recommended**: manually review each Q&A pair. Unreasonable ones can be filtered before running the pipeline.


In [ ]:
# Save generated testset for audit
testset_path = os.path.join(OUTPUT_DIR, "generated_testset.xlsx")
df_generated.to_excel(testset_path, index=False)
print(f"Generated testset saved to: {testset_path}")
print("Review quality — check that each Q&A pair is clinically reasonable.\n")

print("─" * 80)
for i, row in df_generated.iterrows():
    q  = str(row.get(q_col,  ""))
    gt = str(row.get(gt_col, ""))
    print(f"  Q{i+1}: {q[:110]}")
    print(f"  A{i+1}: {gt[:130]}")
    print()


## Cell 14 — Run RAG Pipeline on Generated Questions

Sends auto-generated questions through the RAG pipeline. Results are cached to JSON.


In [ ]:
eval_questions     = df_generated[q_col].dropna().astype(str).str.strip().tolist()
eval_ground_truths = df_generated[gt_col].dropna().astype(str).str.strip().tolist()

# Align lengths (guard against ragas returning fewer rows than requested)
min_len = min(len(eval_questions), len(eval_ground_truths))
eval_questions, eval_ground_truths = eval_questions[:min_len], eval_ground_truths[:min_len]

print(f"Running RAG pipeline on {min_len} auto-generated questions...\n")

eval_records = run_pipeline_batch(
    questions=eval_questions,
    ground_truths=eval_ground_truths,
)

cache_path = os.path.join(OUTPUT_DIR, "pipeline_cache.json")
with open(cache_path, "w", encoding="utf-8") as f:
    json.dump(eval_records, f, indent=2, ensure_ascii=False)
print(f"\nPipeline outputs cached to: {cache_path}")


## Cell 14b — (Optional) Reload Pipeline Cache

If the pipeline has already been run, uncomment this cell to skip re-running it.

In [ ]:
# OPTIONAL — Uncomment to reload Synthetic Evaluation pipeline cache
# cache_path = os.path.join(OUTPUT_DIR, "pipeline_cache.json")
# with open(cache_path, encoding="utf-8") as f:
#     eval_records = json.load(f)
# print(f"Reloaded {len(eval_records)} Synthetic Evaluation records from cache.")


## Cell 15 — RAGAS Evaluation

In [ ]:
ragas_result, eval_detail_df, eval_scores = run_ragas(
    eval_records, label="Synthetic Evaluation — Auto-Generated QA"
)


## Cell 16 — Save Results

In [ ]:
summary_path = os.path.join(OUTPUT_DIR, "evaluation_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(eval_scores, f, indent=2)

detail_path = os.path.join(OUTPUT_DIR, "evaluation_detail.xlsx")
eval_detail_df.to_excel(detail_path, index=False)

print(f"Evaluation results saved:")
print(f"  Summary -> {summary_path}")
print(f"  Detail  -> {detail_path}\n")

score_cols = [c for c in ["faithfulness","answer_relevancy"]
                 if c in eval_detail_df.columns]
q_col_display = "question" if "question" in eval_detail_df.columns else "user_input"

display(

    eval_detail_df[[q_col_display] + score_cols]

    .style.background_gradient(subset=score_cols, cmap="RdYlGn", vmin=0, vmax=1)

)


---
# ═══════════════════════════════════════
# RESULTS & ANALYSIS
# ═══════════════════════════════════════

**Score interpretation guide**:

| Situation | Diagnosis |
|-----------|-----------|
| Low `Faithfulness` | LLM hallucination — strengthen prompt or improve retrieval |
| Low `Answer Relevancy` | LLM is off-topic — check query rewriting or generation prompt |
| Low `Context Precision` | Too many irrelevant documents retrieved — tune TOP_K or embeddings |
| Low `Context Recall` | Required documents not retrieved — increase TOP_K or adjust chunk size |

## Cell 17 — Score Summary Table


In [ ]:
import json, os

with open(os.path.join(OUTPUT_DIR, "evaluation_summary.json")) as f:
    eval_scores = json.load(f)

metrics = ["faithfulness", "answer_relevancy"]

summary_df = pd.DataFrame({
    "Metric":                  metrics,
    "Score (Auto-Generated)": [eval_scores[m] for m in metrics],
})

summary_path = os.path.join(OUTPUT_DIR, "ragas_summary.xlsx")
summary_df.to_excel(summary_path, index=False)
print(f"Summary saved to: {summary_path}\n")

display(
    summary_df.style
    .background_gradient(
        subset=["Score (Auto-Generated)"],
        cmap="RdYlGn", vmin=0, vmax=1,
    )
    .format({"Score (Auto-Generated)": "{:.4f}"})
)

## Cell 18 — RAGAS Score Bar Chart

Shows the four metric scores for the auto-generated evaluation. Suitable for reports and presentations.


In [ ]:
labels_display = ["Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall"]
vals = [eval_scores[m] for m in metrics]
colors = ["#42A5F5", "#66BB6A", "#FFA726", "#EF5350"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels_display, vals, color=colors, edgecolor="white", width=0.5)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f"{val:.3f}", ha="center", va="bottom",
            fontsize=11, fontweight="bold", color="#333")

ax.set_ylim(0, 1.18)
ax.set_ylabel("Score (0–1)", fontsize=11)
ax.set_title("ClinicalRAG — RAGAS Evaluation Results", fontsize=13, fontweight="bold", pad=12)
ax.axhline(y=0.7, color="gray", linestyle="--", linewidth=0.8, alpha=0.5, label="0.7 reference line")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
sns.despine()

bar_path = os.path.join(OUTPUT_DIR, "ragas_bar_chart.png")
plt.tight_layout()
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Bar chart saved to: {bar_path}")


## Cell 19 — Per-Question Score Distribution (Box Plot)

Outliers indicate individual questions where the system underperforms — useful for targeted debugging.


In [ ]:
score_cols = [c for c in ["faithfulness","answer_relevancy"]
              if c in eval_detail_df.columns]

fig, axes = plt.subplots(1, len(score_cols), figsize=(4*len(score_cols), 5), sharey=True)
palette   = ["#42A5F5", "#66BB6A"]

for ax, col, color in zip(axes, score_cols, palette):
    vals = eval_detail_df[col].dropna()
    ax.boxplot(vals, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color="black", linewidth=2))
    ax.scatter([1]*len(vals), vals, alpha=0.65, color=color, s=45, zorder=3)
    ax.set_title(col.replace("_","\n"), fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_xticks([])
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("ClinicalRAG — Per-Question Score Distribution",
             fontsize=13, fontweight="bold", y=1.02)
dist_path = os.path.join(OUTPUT_DIR, "score_distribution.png")
plt.tight_layout()
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Distribution chart saved to: {dist_path}")


## Cell 20 — Final Output Checklist

Verifies that all output files were saved successfully.


In [ ]:
print("=" * 65)
print("  ClinicalRAG RAGAS Evaluation — Output Checklist")
print("=" * 65)

output_files = [
    ("Generated testset",            "generated_testset.xlsx"),
    ("Pipeline cache",               "pipeline_cache.json"),
    ("RAGAS summary (JSON)",         "evaluation_summary.json"),
    ("RAGAS summary (Excel)",        "ragas_summary.xlsx"),
    ("Per-question detail",          "evaluation_detail.xlsx"),
    ("Bar chart",                    "ragas_bar_chart.png"),
    ("Score distribution",           "score_distribution.png"),
]

for desc, fname in output_files:
    full_path = os.path.join(OUTPUT_DIR, fname)
    exists    = os.path.exists(full_path)
    status    = "OK     " if exists else "MISSING"
    print(f"  [{status}]  {desc}")
    if exists:
        print(f"             {full_path}")
    print()

print("=" * 65)
print("  Final Scores")
print("=" * 65)
print(f"  {'Metric':<28} {'Score':>10}")
print("  " + "-" * 40)
for m in metrics:
    v = eval_scores[m]
    print(f"  {m:<28} {v:>10.4f}")
print("=" * 65)


---
## Appendix A — Existing Evaluator vs RAGAS

| Aspect | `src/evaluator.py` (pipeline-internal) | RAGAS (this notebook) |
|--------|----------------------------------------|-----------------------|
| Evaluation timing | During pipeline execution (real-time gatekeeper) | After pipeline execution (external judge) |
| Decision logic | Rule-based: content length > 80 chars, rerank score ≥ 0.20 | LLM-as-judge: GPT verifies each claim |
| Hallucination detection | Not possible | Faithfulness metric |
| Requires ground truth | No | Yes (auto-generated by TestsetGenerator) |
| Compute cost | Very low (local) | Moderate (OpenAI API calls) |
| Recommended use | Real-time retrieval quality gate | Rigorous evaluation during development |

**Conclusion**: The two approaches are complementary — use both together for best results.

---
## Appendix B — Frequently Asked Questions

**Q1: Getting RateLimitError?**  
Increase `SLEEP_BETWEEN_CALLS` to 3–5 seconds.

**Q2: TestsetGenerator producing poor-quality questions?**  
This is normal. Manually filter out unreasonable pairs, or increase `TESTSET_SIZE` for a more stable average.

**Q3: Why does the notebook only report two metrics?**  
This portfolio version reports only faithfulness and answer_relevancy to match the live app display and keep evaluation focused.

**Q4: FAISS index does not exist?**  
Run `build_index()` in `src/ingest.py` first (one-time setup step).

**Q5: discharge.csv has different column names?**  
The load cell attempts automatic column detection. You can also manually set `text_col` to the correct column name.
